# Mini Lab 0 – Clustering Warm-Up

## 0. Objectives

In this mini lab you will:
1. Visualize and run basic clustering on simple 2D data.
2. Implement a minimal K-means from scratch.
3. Compare K-means, hierarchical clustering, and DBSCAN on non-globular data.
4. Do a tiny text-clustering exercise with TF-IDF.
5. Use basic internal validity measures (especially silhouette).

---

## 1. Toy 2D data + “black box” K-means

**Goal:** get an intuitive feeling for K-means on an easy dataset.

3 datasets:  
1.	2D blobs dataset (for Section 1 and 2 of the mini-lab)  
Download: mini_lab0_blobs2d.csv → Download it￼  
- Columns:
	- x1, x2: 2D coordinates
   - true_label: integer 0–3 (ground-truth blob, you can ignore in unsupervised steps)

2.	2D “two moons” dataset (for Section 3 + DBSCAN / hierarchical)  
Download: mini_lab0_moons2d.csv → Download it￼  
- Columns:
   - x1, x2: 2D coordinates
   - true_label: 0 or 1 (which moon)

3.	Tiny text corpus (for Section 4, TF-IDF + clustering)
Download: mini_lab0_text_corpus.csv → Download it￼  
- Columns:
	- text: short document
	- topic_hint: rough topic (tech, sports, food, mixed) – use only for sanity checks, not in clustering.

---

> 1. Plot the data in 2D:  
>    - Single color (pretend you don’t know any labels).

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv('../../Dataset/LAB8/mini_lab0_blobs2d.csv', sep=',')
display(df)

In [ ]:
x1 = df['x1']
x2 = df['x2']

plt.figure(figsize=(8,8))
sns.scatterplot(x=x1, y=x2)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title("Let's explore the data distribution")
plt.show()

Yep, the data distribution makes sense, it's not random, we can already identify clusters!  
--> clustering makes sense

We can identify 4 clusters

> 2. Run K-means from a library:  
>    - Use a value of K that matches the number of clusters you generated.  
>    - Obtain the cluster labels and centroids.

---

# KMeans implementation
**1. What you need before K-means**  

You need a data matrix:
- Shape: (n_samples, n_features)
- For your toy 2D dataset:
	- n_samples ≈ 300
	- n_features = 2 (x and y coordinates)
Call it X. That’s literally the only required input.

**2. Build a KMeans model object**  
```python
class sklearn.cluster.KMeans(n_clusters=8, *, init='k-means++', n_init='auto', max_iter=300, tol=0.0001, verbose=0, random_state=None, copy_x=True, algorithm='lloyd')
```

**3. fit the model your matrix**  
```python
kmeans.fit(X)
```

**4. get out of the model what you need**  
For example:
```python
kmeans.labels_
kmeans.cluster_centers_
kmeans.inertia_
```
- labels = labels_: An array that tells you, for each point, which cluster it belongs to.
   - example: array([0, 0, 2, 1, 1, 1, 2, 0, 0, 2]), which means:
      - Point 0 → cluster 0
      - Point 1 → cluster 0
      - Point 2 → cluster 2
      - Point 3 → cluster 1

- centroids = cluster_centers_: an array of the coordinates of the K centroids (one per cluster).
- Shape: (K, n_features) --> so for example if I ask KMeans to identify 4 clusters it will return 4 centroids and each centroids will have coordinates, as many coordinates as many features the DS has.
   - exmaple:
```python
# array([
#   [ 1.2,  3.4],   # centroid of cluster 0
#   [-0.5,  2.1],   # centroid of cluster 1
#   [ 4.0, -1.7]    # centroid of cluster 2
# ])
```

- SSE = inertia_: The total Sum of Squared Errors (SSE) for the final clustering.
- it's just one number

Its purpose depends on how you use it:  
1. For one run of K-means:  
It tells you how tight the clustering is (sum of squared distances to centroids).  

2. For multiple K values (elbow method):  
```python
inertias = []
for K in [2, 3, 4, 5]:
    km = KMeans(n_clusters=K, ...)
    km.fit(X)
    inertias.append(km.inertia_)
```
Then you plot K vs inertia and look for the “elbow”.

Build the matrix you'll feed the KMeans

In [ ]:
# build matrix to feed to KMeans

    # this is ass
ass = False
if ass:
    df_data = df.drop(columns='true_label')

    x = []
    for row in df_data.values:
        r = []
        for v in row:
            v = np.array(v)
            r.append(v)
        x.append(r)
    print(np.array(x))

    # by the way what you did is identical to:
    print(df_data.values)


# very nice
otherwise = True
if otherwise:
    # df['x1', 'x2'] ---> NOPE:
        # df['x1']          # Series
        # df[['x1', 'x2']]  # DataFrame
    
    # very useful
    # print(df[['x1', 'x2']])
    x = df[['x1', 'x2']].to_numpy()
    print(x)


# ok too
otherwise2 = False
if otherwise2:
    x = df.drop(columns='true_label')
    x = x.to_numpy()
    print(x)


> build the KMeans, feed it and get parameters

In [ ]:
KM_clustering = KMeans(n_clusters=4, random_state=42)

KM_clustering.fit(x)

labels = KM_clustering.labels_
centroids = KM_clustering.cluster_centers_
SSE = KM_clustering.inertia_

print(labels, '\n', centroids, '\n', SSE)

> 3. Plot the clustered data:  
>   - Color points by K-means labels.  
>   - Add the centroids as special markers.

In [ ]:
plt.figure(figsize=(8,8))
col_1 = x[:, 0]             # take all rows, take first column
col_2 = x[:, 1]             # take all rows, take second column


# KMeans associated to each point a label, which is the cluster that point belongs to, in the plot we want to plot the point x1,x2 and assign it a colour depending on the cluster it belongs to
sns.scatterplot(x=col_1, y=col_2, hue=labels)
sns.scatterplot(x = centroids[:, 0], y=centroids[:, 1])
plt.show()

> 4. Change K (e.g. try 2, then 3, then 5):  
>    - For each K, run K-means and compute the silhouette score.  
>    - Compare visually and via silhouette which K “makes more sense” for this toy data.

NAH, I'll just compute the silhoutte score of my clustering:  
> - sklearn.metrics.silhouette_score(X, labels, *, metric='euclidean', sample_size=None, random_state=None, **kwds)  

**X**: {array-like, sparse matrix} of shape (n_samples_a, n_samples_a) if metric == “precomputed” or (n_samples_a, n_features) otherwise. An array of pairwise distances between samples, or a feature array.  

**labels**:  array-like of shape (n_samples,). Predicted labels for each sample.  

In [ ]:
silhouette = silhouette_score(X=x, labels=labels)
print(silhouette)

# Actually this is enough, just go do the lab my man